In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r"C:\Users\eider\OneDrive\Escritorio\Reto_07_Morado\Code\datos_prestamos_limpios.csv")

df["Interes_Anual"] = df["Ratio_Interes"] / 100
df["i_mensual"] = df["Interes_Anual"] / 12
df["Ingresos_mensuales"] = df["Ingresos"] / 12

df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima,Interes_Anual,i_mensual,Ingresos_mensuales
0,S97R7X,18,16000.0,5000.0,397,19,1,8.06,48,0.10,...,Soltero,1,0,Automóvil,0,0,15.0,0.0806,0.006717,1333.333333
1,T3ZE0N,69,72673.0,32340.0,784,320,2,15.04,48,0.12,...,Casado,0,1,Educación,0,0,15.0,0.1504,0.012533,6056.083333
2,RLGTBY,50,62116.0,37278.0,486,217,3,21.96,12,0.55,...,Casado,1,0,Automóvil,1,1,15.0,0.2196,0.018300,5176.333333
3,BZ86CV,64,59846.0,19784.0,308,340,1,24.26,12,0.35,...,Divorciado,1,0,Negocios,1,0,15.0,0.2426,0.020217,4987.166667
4,5OD75M,62,28413.0,13751.0,412,476,2,5.73,36,0.14,...,Casado,0,0,Negocios,0,0,15.0,0.0573,0.004775,2367.750000


In [3]:
def amortizacion_frances(capital, i, n):
    cuota = capital * (i * (1 + i)**n) / ((1 + i)**n - 1)
    saldo = capital
    intereses_totales = 0
    
    for _ in range(n):
        intereses = saldo * i
        amortizacion = cuota - intereses
        saldo -= amortizacion
        intereses_totales += intereses
        
    return cuota, intereses_totales

In [4]:
def amortizacion_aleman(capital, i, n):
    amortizacion_constante = capital / n
    saldo = capital
    intereses_totales = 0
    
    for _ in range(n):
        intereses = saldo * i
        saldo -= amortizacion_constante
        intereses_totales += intereses
        
    cuota_inicial = amortizacion_constante + capital * i
    
    return cuota_inicial, intereses_totales

In [5]:
cuota_frances = []
intereses_frances = []
cuota_aleman = []
intereses_aleman = []

for _, row in df.iterrows():
    
    C = row["Monto_Inicial"]
    i = row["i_mensual"]
    n = int(row["Duracion"])
    
    cf, int_f = amortizacion_frances(C, i, n)
    ca, int_a = amortizacion_aleman(C, i, n)
    
    cuota_frances.append(cf)
    intereses_frances.append(int_f)
    
    cuota_aleman.append(ca)
    intereses_aleman.append(int_a)

df["Cuota_Frances"] = cuota_frances
df["Intereses_Totales_Frances"] = intereses_frances
df["Cuota_Aleman_Inicial"] = cuota_aleman
df["Intereses_Totales_Aleman"] = intereses_aleman

df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Fiador,Impago,Prima,Interes_Anual,i_mensual,Ingresos_mensuales,Cuota_Frances,Intereses_Totales_Frances,Cuota_Aleman_Inicial,Intereses_Totales_Aleman
0,S97R7X,18,16000.0,5000.0,397,19,1,8.06,48,0.10,...,0,0,15.0,0.0806,0.006717,1333.333333,122.205481,865.863106,137.750000,822.791667
1,T3ZE0N,69,72673.0,32340.0,784,320,2,15.04,48,0.12,...,0,0,15.0,0.1504,0.012533,6056.083333,900.702270,10893.708978,1079.078000,9930.536000
2,RLGTBY,50,62116.0,37278.0,486,217,3,21.96,12,0.55,...,1,1,15.0,0.2196,0.018300,5176.333333,3488.293687,4581.524239,3788.687400,4434.218100
3,BZ86CV,64,59846.0,19784.0,308,340,1,24.26,12,0.35,...,1,0,15.0,0.2426,0.020217,4987.166667,1873.257267,2695.087199,2048.633200,2599.782467
4,5OD75M,62,28413.0,13751.0,412,476,2,5.73,36,0.14,...,0,0,15.0,0.0573,0.004775,2367.750000,416.651865,1248.467154,447.633247,1214.728962


In [6]:
#Metricas de riesgo
df["Ratio_Frances_Ingresos"] = df["Cuota_Frances"] / df["Ingresos_mensuales"]
df["Ratio_Aleman_Ingresos"] = df["Cuota_Aleman_Inicial"] / df["Ingresos_mensuales"]

df[["Ratio_Frances_Ingresos", "Ratio_Aleman_Ingresos"]].describe()

,Ratio_Frances_Ingresos,Ratio_Aleman_Ingresos
count,255347.000000,255347.000000
mean,0.580384,0.645173
std,0.658187,0.707482
min,0.012254,0.013296
25%,0.172557,0.197086
50%,0.357104,0.406293
75%,0.728996,0.821810
max,7.775609,7.962543


In [7]:
# CRITERIO DE ASIGNACIÓN

# Diferencia de coste entre sistemas (absoluta)
df["Diferencia_Intereses"] = (
    df["Intereses_Totales_Frances"] - df["Intereses_Totales_Aleman"]
)

# Diferencia relativa respecto al capital
df["Diferencia_Relativa"] = (
    df["Diferencia_Intereses"] / df["Monto_Inicial"]
)

# 3Umbral de solvencia (percentil 70 del scoring)
scoring_umbral = df["Scoring_Crediticio"].quantile(0.70)

# Condición para sistema francés
condicion_frances = (
    (df["Ratio_Frances_Ingresos"] <= 0.50) &         
    (df["Scoring_Crediticio"] >= scoring_umbral) &  
    ((df["Diferencia_Relativa"] <= 0.03))              
)

# 5Asignación final
df["Sistema_Asignado"] = np.where(condicion_frances, "Frances", "Aleman")

# Distribución final
df["Sistema_Asignado"].value_counts()

Sistema_Asignado
Aleman     225813
Frances     29534
Name: count, dtype: int64

In [8]:
# Impacto real tras asignación

intereses_reales = np.where(
    df["Sistema_Asignado"] == "Frances",
    df["Intereses_Totales_Frances"],
    df["Intereses_Totales_Aleman"]
)

df["Intereses_Finales_Cartera"] = intereses_reales

print(f"Interés promedio real tras asignación: {df['Intereses_Finales_Cartera'].mean():.2f} €")

Interés promedio real tras asignación: 9293.13 €


In [9]:
ahorro_vs_frances = df["Intereses_Totales_Frances"].mean() - df["Intereses_Finales_Cartera"].mean()

print(f"Ahorro promedio frente a aplicar solo sistema francés: {ahorro_vs_frances:.2f} €")

Ahorro promedio frente a aplicar solo sistema francés: 874.99 €


In [10]:
tabla_comparacion = pd.DataFrame({
    "Sistema": ["Frances", "Aleman"],
    "Intereses_Promedio": [
        df["Intereses_Totales_Frances"].mean(),
        df["Intereses_Totales_Aleman"].mean()
    ],
    "Cuota_Promedio": [
        df["Cuota_Frances"].mean(),
        df["Cuota_Aleman_Inicial"].mean()
    ]
})

tabla_comparacion

,Sistema,Intereses_Promedio,Cuota_Promedio
0,Frances,10168.125433,1974.056513
1,Aleman,9262.662039,2194.282669


In [26]:
df[[
    "Monto_Inicial",
    "Duracion",
    "Cuota_Frances",
    "Cuota_Aleman_Inicial",
    "Intereses_Totales_Frances",
    "Intereses_Totales_Aleman",
    "Sistema_Asignado"
]].head()

,Monto_Inicial,Duracion,Cuota_Frances,Cuota_Aleman_Inicial,Intereses_Totales_Frances,Intereses_Totales_Aleman,Sistema_Asignado
0,5000.0,48,122.205481,137.750000,865.863106,822.791667,Aleman
1,32340.0,48,900.702270,1079.078000,10893.708978,9930.536000,Frances
2,37278.0,12,3488.293687,3788.687400,4581.524239,4434.218100,Aleman
3,19784.0,12,1873.257267,2048.633200,2695.087199,2599.782467,Aleman
4,13751.0,36,416.651865,447.633247,1248.467154,1214.728962,Aleman


In [27]:
riesgo_segmentos = df.groupby("Segmento_Riesgo").agg({
    "Monto_Inicial": "mean",
    "Cuota_Frances": "mean",
    "Ratio_Frances_Ingresos": "mean"
})

riesgo_segmentos

C:\Users\eider\AppData\Local\Temp\ipykernel_21652\981372569.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  riesgo_segmentos = df.groupby("Segmento_Riesgo").agg({


,Monto_Inicial,Cuota_Frances,Ratio_Frances_Ingresos
Segmento_Riesgo,,,
Bajo,18660.007291,570.389990,0.158867
Medio,38466.235150,1364.875777,0.390528
Alto,60326.943744,2384.574933,0.707593
Muy alto,100013.279520,5924.448693,1.769381


In [28]:
pd.crosstab(df["Segmento_Riesgo"], df["Sistema_Asignado"])

Sistema_Asignado,Aleman,Frances
Segmento_Riesgo,,
Bajo,92468,18624
Medio,36715,10910
Alto,55589,0
Muy alto,41041,0


In [29]:
prob_impago = df["Impago"].mean()

df["Riesgo_Frances"] = prob_impago * df["Monto_Inicial"]

In [31]:
df[["Monto_Inicial", "Riesgo_Frances"]].head()

,Monto_Inicial,Riesgo_Frances
0,5000.0,580.641245
1,32340.0,3755.587573
2,37278.0,4329.028867
3,19784.0,2297.481278
4,13751.0,1596.879552


OBJETIVO 2

In [17]:
# VALOR PRESENTE DE LA CARTERA (objetivo 2)

df["k"] = (df["Duracion"] / 2).astype(int)
df["n_restante"] = df["Duracion"] - df["k"]

df["Cuota_Real"] = np.where(
    df["Sistema_Asignado"] == "Frances",
    df["Cuota_Frances"],
    df["Cuota_Aleman_Inicial"]
)

i = df["i_mensual"]

df["VP_Cartera"] = np.where(
    i == 0,
    df["Cuota_Real"] * df["n_restante"],
    df["Cuota_Real"] * (1 - (1+i)**(-df["n_restante"])) / i
)

print("Valor presente medio cartera:",
      round(df["VP_Cartera"].mean(),2),"€")

print("Valor presente total cartera:",
      round(df["VP_Cartera"].sum(),2),"€")

Valor presente medio cartera: 27653.97 €
Valor presente total cartera: 7061359173.62 €


In [18]:
# SEGMENTACIÓN DE CLIENTES SEGÚN RATIO DE ENDEUDAMIENTO

df["Segmento_Riesgo"] = pd.cut(
    df["Ratio_Frances_Ingresos"],
    bins=[0,0.3,0.5,1,np.inf],
    labels=["Bajo","Medio","Alto","Muy alto"]
)

print("Distribución de clientes por riesgo:")
print(df["Segmento_Riesgo"].value_counts())

Distribución de clientes por riesgo:
Segmento_Riesgo
Bajo        111092
Alto         55589
Medio        47625
Muy alto     41041
Name: count, dtype: int64


In [19]:
analisis_segmentos = df.groupby("Segmento_Riesgo").agg({
    "Monto_Inicial":"mean",
    "Cuota_Frances":"mean",
    "VP_Cartera":"mean"
})

analisis_segmentos

C:\Users\eider\AppData\Local\Temp\ipykernel_21652\3735778111.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analisis_segmentos = df.groupby("Segmento_Riesgo").agg({


,Monto_Inicial,Cuota_Frances,VP_Cartera
Segmento_Riesgo,,,
Bajo,18660.007291,570.389990,11844.526172
Medio,38466.235150,1364.875777,24183.808026
Alto,60326.943744,2384.574933,38285.018269
Muy alto,100013.279520,5924.448693,60075.225612


In [20]:
pd.crosstab(df["Segmento_Riesgo"], df["Sistema_Asignado"])

Sistema_Asignado,Aleman,Frances
Segmento_Riesgo,,
Bajo,92468,18624
Medio,36715,10910
Alto,55589,0
Muy alto,41041,0
